In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv, find_dotenv

from doc_chat.evaluation.rag_judge_agent import evaluate_response
from doc_chat.llm import create_llm
from doc_chat.rag.citation_retrieval_chain import CitationRetrievalChain
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore

_ = load_dotenv(find_dotenv())


In [ ]:
pd_documents = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/documents_processed.csv')
pd_documents

In [ ]:
# Load and split the documents
all_splits = []
for index, row in pd_documents.iterrows():
    doc_path = f"../../data/single_topic_rag_evaluation_dataset/processed/pdf/document_{index}.pdf"
    loader = DocumentLoader(doc_path)
    documents, splits = loader.load_and_split()
    pd_documents.loc[index, 'num_splits'] = int(len(splits))
    all_splits.append(splits)

pd_documents['num_splits'] = pd_documents['num_splits'].astype(int)
pd_documents

In [ ]:
# create vector store
chroma_persist_directory = '/tmp/doc-chat-eval/vectorstore'
if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'single_topic_rag_evaluation'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory, embedding_model='text-embedding-3-small')

In [ ]:
# Index the documents
for index, splits in enumerate(all_splits):
      print('indexing document: ', index, 'number of splits: ', len(splits))
      # enable this to index the documents
      # vector_store.create_document_collection(user_id=user_id,
      #                                         collection_id=f'document_{index}',
      #                                         document_splits=splits,
      #                                         file_name=f'document_{index}')

In [ ]:
# create chain for retrieval
retrieval_chain = CitationRetrievalChain(retriever=None,
                                         # TODO: use vector store as retriever
                                         llm=create_llm())

In [ ]:
# create function to retrieve documents and answer questions with the chain
def retrieve_and_answer(data):
    for index, question_row in data.iterrows():
        question = question_row['question']
        document_index = question_row['document_index']
        collection_id = f'document_{document_index}'

        print(f'Processing question: {index}, collection_id: {collection_id}')

        # Retrieve documents from the vector store
        found_documents = vector_store.retrieve_documents(user_id=user_id, collection_id=collection_id, query=question, k=5)

        # generate answer using the retrieval chain
        response = retrieval_chain.invoke(query=question, documents=found_documents, include_references_in_answer=False)

        # store the answer in the dataframe
        data.loc[index, 'generated_answer'] = response['answer']
        data.loc[index, 'referenced_documents'] = json.dumps(response['referenced_documents'])

    return data



In [ ]:
# create the results directory if it does not exist
results_directory = '../../data/single_topic_rag_evaluation_dataset/results'
if not os.path.exists(results_directory):
    os.makedirs(results_directory)

pd_documents.to_csv('../../data/single_topic_rag_evaluation_dataset/results/documents_predict.csv', index=False)

In [ ]:
# single passage questions
pd_single_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/single_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_single_passage)
pd_result.to_csv('../../data/single_topic_rag_evaluation_dataset/results/single_passage_answer_questions_predict.csv', index=False)

# multi passage questions
pd_multi_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/multi_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_multi_passage)
pd_multi_passage.to_csv('../../data/single_topic_rag_evaluation_dataset/results/multi_passage_answer_questions_predict.csv', index=False)

# no answer questions
pd_no_answer = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/no_answer_questions.csv')
pd_result = retrieve_and_answer(pd_no_answer)
pd_no_answer.to_csv('../../data/single_topic_rag_evaluation_dataset/results/no_answer_questions_predict.csv', index=False)

In [2]:
# evaluate the results with LLM acting as judge

pd_single_passage = pd.read_csv(
    '../../data/single_topic_rag_evaluation_dataset/results/single_passage_answer_questions_predict.csv')

result = evaluate_response(
    instruction=pd_single_passage.loc[0, 'question'],
    reference_answer=pd_single_passage.loc[0, 'answer'],
    generated_answer=pd_single_passage.loc[0, 'generated_answer']
)

result

Evaluating response...
Instruction: What do keybullet kin drop? Reference Answer: Keybullet kin drop a key upon death. Generated Answer: Keybullet Kin drop a key upon death.


EvaluationFeedback(feedback='The response "Keybullet Kin drop a key upon death." is completely correct, accurate, and factual based on the reference answer. It matches the reference answer exactly, indicating that the information provided is precise and aligns perfectly with the expected response.', score=5)